In [17]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI 
from rich import print
from langgraph.checkpoint.memory import MemorySaver

from typing import TypedDict, Annotated 


Annotated allows you to:

Add extra metadata (additional information) to a type.

Annotated[actual_type, extra_info]

age: Annotated[int, "Age must be positive"]

Meaning:
Actual type = int
Extra metadata = "Age must be positive"

In [18]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    messages : Annotated[list[BaseMessage],add_messages]

In [20]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')
def Chat_node(state:ChatState):
    
    # Take message form user
    message = state['messages']

    # send to llm
    response = llm.invoke(message)

    # return response
    return {'messages':[response]}


In [21]:
checkpointer = MemorySaver()

# Initial Graph
graph = StateGraph(ChatState)

# Add nodes
graph.add_node('Chat_node',Chat_node)

# add edges
graph.add_edge(START,'Chat_node')
graph.add_edge('Chat_node',END)

# COmpile graph

chat_bot = graph.compile(checkpointer=checkpointer)

In [22]:
# initail_state = {
#     'messages':[HumanMessage(content='Who is the CM of maharashtra')]
#     }

# result = chat_bot.invoke(initail_state)['messages'][-1].content
# print(result)

In [23]:
thread_id = '1'

while True:

    user_message = input("Ask anythings.....")
    print('User messgae :', user_message)
    if user_message.strip().lower() in ['exit','bye','quit']:
        break

    config = {'configurable':{'thread_id':thread_id}}
    chat_messgae = chat_bot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)

    print('AI message :',chat_messgae['messages'][-1].content)

User messgae : hii my name is nikhil

AI message : Hi Nikhil! It's nice to meet you. How can I help you today?

User messgae : now tell me whats my name

AI message : Your name is Nikhil.

User messgae : exit

In [25]:
print(chat_bot.get_state(config=config))

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='hii my name is nikhil',
                additional_kwargs={},
                response_metadata={},
                id='3b610e9b-a3e0-46fc-a7e2-aed711944477'
            ),
            AIMessage(
                content="Hi Nikhil! It's nice to meet you. How can I help you today?",
                additional_kwargs={},
                response_metadata={
                    'finish_reason': 'STOP',
                    'model_name': 'gemini-2.5-flash-lite',
                    'safety_ratings': [],
                    'model_provider': 'google_genai'
                },
                id='lc_run--019dfe5d-22b7-7412-bbd0-957bdbfc24a7-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 7,
                    'output_tokens': 18,
                    'total_tokens': 25,
                    'input_token_details': {'cache_read': 0}
                }
            ),
            HumanMessage(
                content='now tell me whats my name',
                additional_kwargs={},
                response_metadata={},
                id='01b1a3f6-33b0-4a33-b99a-21febd1c09e5'
            ),
            AIMessage(
                content='Your name is Nikhil.',
                additional_kwargs={},
                response_metadata={
                    'finish_reason': 'STOP',
                    'model_name': 'gemini-2.5-flash-lite',
                    'safety_ratings': [],
                    'model_provider': 'google_genai'
                },
                id='lc_run--019dfe5d-5576-7ab3-a02e-b84e53106971-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 33,
                    'output_tokens': 5,
                    'total_tokens': 38,
                    'input_token_details': {'cache_read': 0}
                }
            )
        ]
    },
    next=(),
    config={
        'configurable': {
            'thread_id': '1',
            'checkpoint_ns': '',
            'checkpoint_id': '1f149723-4361-6ba5-8004-557a1df0e51c'
        }
    },
    metadata={'source': 'loop', 'step': 4, 'parents': {}},
    created_at='2026-05-06T17:37:07.625647+00:00',
    parent_config={
        'configurable': {
            'thread_id': '1',
            'checkpoint_ns': '',
            'checkpoint_id': '1f149723-15b3-63ae-8003-569741504567'
        }
    },
    tasks=(),
    interrupts=()
)